In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "font.family": "serif",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "figure.titlesize": 16,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,   # good for illustrator
    "ps.fonttype": 42
})

# ------------------------------------------------------------
# MOCK DATA
# ------------------------------------------------------------
n_sentences = 20
moral_rdm = np.abs(np.random.randn(n_sentences, n_sentences))
np.fill_diagonal(moral_rdm, 0)

ispc_rdms = {
    "L STG/MTG": np.random.rand(n_sentences, n_sentences),
    "R STG/MTG": np.random.rand(n_sentences, n_sentences),
    "R TPJ": np.random.rand(n_sentences, n_sentences)
}
for rdm in ispc_rdms.values():
    np.fill_diagonal(rdm, 0)

# Spearman correlations 
rho_vals = {
    "L STG/MTG": 0.21,
    "R STG/MTG": 0.34,
    "R TPJ": 0.29
}

# Permutation null distributions for each ROI 
perm_null = {
    roi: np.random.randn(1000) * 0.1
    for roi in ispc_rdms.keys()
}

# Compute permutation p-values
p_values = {
    roi: np.mean(np.abs(perm_null[roi]) >= abs(rho_vals[roi]))
    for roi in ispc_rdms.keys()
}

# ------------------------------------------------------------
# FIGURE LAYOUT
# ------------------------------------------------------------
fig = plt.figure(figsize=(16, 15))
gs = gridspec.GridSpec(
    4, 2, 
    height_ratios=[1, 1, 1.2, 1.2],
    hspace=0.4, wspace=0.25
)

# ------------------------------------------------------------
# PANEL A: Sentence-locked pattern extraction
# ------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("A. Sentence-locked Multi-Voxel Pattern Extraction")

TRs = np.arange(0, 40)
signal = np.sin(TRs / 4) + np.random.randn(len(TRs)) * 0.2
ax1.plot(TRs, signal, lw=2, color="black")

event_onset, event_offset = 10, 15
shift = 3
ax1.axvspan(event_onset + shift, event_offset + shift, color="gold", alpha=0.3)
ax1.text(event_onset + shift + 0.5, max(signal) * 0.8,
         "HRF-shifted\nsentence window",
         fontsize=10, color="black")

ax1.set_xlabel("Time (TR)")
ax1.set_ylabel("BOLD signal")

sns.despine(ax=ax1)

# ------------------------------------------------------------
# PANEL B: Leave-one-participant-out diagram
# ------------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("B. Leave-One-Participant-Out Group Templates")

participants = np.arange(1, 7)
ax2.scatter(participants[:-1], [1]*5, s=250, color="grey")
ax2.scatter([participants[-1]], [1], s=250, color="crimson")

for p in participants[:-1]:
    ax2.text(p, 1.03, f"P{p}", ha='center', fontsize=10)
ax2.text(participants[-1], 1.03, f"P{participants[-1]}", ha="center", color="crimson", fontsize=10)

ax2.set_ylim(0.9, 1.1)
ax2.set_yticks([])
ax2.set_xticks([])
sns.despine(ax=ax2, left=True, bottom=True)

# ------------------------------------------------------------
# PANEL C: Moral-content RDM
# ------------------------------------------------------------
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_title("C. Moral-content RDM")

sns.heatmap(
    moral_rdm, 
    cmap="viridis", 
    ax=ax3,
    cbar_kws={"shrink": 0.7},
    square=True
)
ax3.set_xlabel("Sentence")
ax3.set_ylabel("Sentence")

# ------------------------------------------------------------
# PANEL D: ISPC-RDM example
# ------------------------------------------------------------
example_roi = "R TPJ"
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_title(f"D. ISPC-RDM ({example_roi})")

sns.heatmap(
    ispc_rdms[example_roi], 
    cmap="rocket", 
    ax=ax4,
    cbar_kws={"shrink": 0.7},
    square=True
)
ax4.set_xlabel("Sentence")
ax4.set_ylabel("Sentence")

# ------------------------------------------------------------
# PANEL E: Spearman correlations (bar plot)
# ------------------------------------------------------------
ax5 = fig.add_subplot(gs[2, :])
ax5.set_title("E. Neural–Moral RDM Spearman Correlation")

rois = list(rho_vals.keys())
vals = list(rho_vals.values())

ax5.bar(rois, vals, color="black", alpha=0.8)
ax5.set_ylabel("Spearman ρ")
ax5.axhline(0, color="black", lw=1)

for i, v in enumerate(vals):
    ax5.text(i, v + 0.02, f"ρ={v:.2f}", ha='center')

sns.despine(ax=ax5)

# ------------------------------------------------------------
# PANEL F: Permutation Null Distribution + P-values
# ------------------------------------------------------------
ax6 = fig.add_subplot(gs[3, :])
ax6.set_title("F. Permutation-based Significance (1000 permutations)")

# violin plots of null distributions
sns.violinplot(
    data=[perm_null[roi] for roi in rois],
    orient="v",
    inner="box",
    palette=["grey"]*len(rois),
    ax=ax6
)

# overlay observed correlations
for i, roi in enumerate(rois):
    ax6.scatter(i, rho_vals[roi], color="crimson", s=80, zorder=3)
    ax6.text(i, rho_vals[roi] + 0.015, f"p={p_values[roi]:.003f}", 
             ha="center", fontsize=10, color="crimson")

ax6.set_xticklabels(rois)
ax6.set_ylabel("Null distribution (Spearman ρ)")
sns.despine(ax=ax6)

# ------------------------------------------------------------
plt.tight_layout()
plt.show()
